---
title: Tools to profile, customize, and inspect experiments
description: Use Qiskit and IBM Quantum Compute Service developer tools to profile, customize, and inspect your experiment settings and runtime environment.
---

# Tools to profile, customize, and inspect experiments

*Usage estimate: 10 minutes on a Heron r2 processor (NOTE: This is an estimate only. Your runtime might vary.)*

Getting good results from a quantum computer takes more than a well-designed circuit. The device's current condition, how your circuit is scheduled, the noise it experiences, and when and how your job runs on the quantum processing unit (QPU) all affect the quality of your results. Qiskit and IBM Quantum&reg; Compute Service offer a growing set of developer tools that expose this information. You can use these tools to act on that information before you commit QPU time to a full experiment, and to debug a finished experiment.

This tutorial walks through a single end-to-end experiment and shows how to combine these tools at each stage of the [Qiskit patterns](/docs/guides/intro-to-patterns) workflow. You refresh device properties to select a better layout, inspect circuit timing, learn a noise model and use it to preview the benefit of error mitigation with local simulation, estimate QPU usage, recalibrate readout at the start of a session, and finally inspect how your job ran on the hardware. Not every tool is useful for every experiment, so choose the tools that address the questions you have about your own workload.

The workflow is as follows:

1. Prepare a mirror circuit of a Trotterized time evolution of a linear transverse-field Ising model.
2. Select a backend with an alternative calibration data set designed for noise-model-based error mitigation methods.
3. Run device benchmarking experiments so that the transpiler uses up-to-date device properties when selecting a layout.
4. Inspect the timing information of the scheduled circuit.
5. Learn Pauli-Lindblad and readout noise models for the circuit.
6. Perform noiseless, unmitigated, and mitigated simulations by using the learned noise model. This shows how much improvement you might gain from the selected error mitigation methods: probabilistic error cancellation (PEC) and Twirled Readout Error eXtinction (TREX) in this tutorial.
7. Get a usage estimate without QPU execution to see whether the experiment needs adjusting to fit your budget.
8. Start a session and trigger calibration so that the experiment uses the most recent calibration data.
9. Run the experiment on the QPU by using Estimator.
10. Inspect the execution spans to see whether there are significant gaps between groups of executions (chunks).

## Requirements

Before starting this tutorial, ensure that you have the following installed:

- Qiskit SDK v2.3 or later, with [visualization](/docs/api/qiskit/visualization) support
- `qiskit-ibm-runtime` v0.52 or later (`pip install qiskit-ibm-runtime`)
- Qiskit Aer (`pip install qiskit-aer`)
- Plotly (`pip install plotly`)
- [Qiskit Device Benchmarking](https://github.com/qiskit-community/qiskit-device-benchmarking) (`pip install git+https://github.com/qiskit-community/qiskit-device-benchmarking.git`)

<Admonition type="note" title="Plan requirements">
Some steps in this tutorial are not available on every plan:

- Dry-run jobs (Steps 2.4 and 3.4) are available only on the Flex, Premium, and On-Prem plans.
- Sessions (Step 4.1) are not available on the Open Plan. To run in job mode instead, set `run_calibration = False` in Step 4.1.
</Admonition>

## Summary of tools used

| Tool | What it tells you or does | Step | Docs |
|---|---|---|---|
| Specially calibrated instructions | Non-standard calibrations that can benefit some experiments. | 2.1 | [Documentation](/docs/guides/custom-calibration) |
| Qubit quality benchmarking | Refreshes device properties data. | 2.2 | [Tutorial](/docs/tutorials/refresh-backend-properties-with-real-time-benchmarking) |
| Timing for scheduled circuits | Visualizes when each instruction runs, to help spot idle time and inefficient instruction alignment. | 2.4 | [Documentation](/docs/guides/qiskit-runtime-circuit-timing) |
| `select-error-mitigation` AI agent skill | Compares error mitigation methods for your workload and reports which ones to use and why. | 3.1 | [GitHub repository](https://github.com/Qiskit/skills) |
| Client-side primitives | New implementations of the primitives that give more insight into, and control over, pre- and post-processing. | 3.1 | [Documentation](/docs/guides/migrate-to-client-side-primitives) |
| `NoiseLearnerV3` | Performs noise learning using both the Pauli-Lindblad and TREX protocols. | 3.2 | [Documentation](/docs/guides/noise-learning) |
| Local testing mode | Verifies syntactic correctness and compares the effects of different error mitigation methods. | 3.3 | [Documentation](/docs/guides/local-testing-mode) |
| Usage estimation | Provides an accurate usage estimate before hardware execution. | 3.4 | [Documentation](/docs/guides/dry-run-jobs) |
| Calibration in sessions | Triggers calibration to ensure the experiment uses optimal readout parameters. | 4.1 | [Documentation](/docs/guides/calibration-jobs) |
| Execution spans | Shows when each circuit configuration ran on the QPU, and in which order. | 4.3 | [Documentation](/docs/guides/sampler-input-output#view-execution-spans) |

## Step 1: Specify a circuit and observable

Prepare a mirror circuit of a Trotterized time evolution of a linear transverse-field Ising model. Mirror the circuit to cancel the circuit unitary and form an identity circuit. Because the ideal output is the all-zero state, the ideal expectation value of each single-qubit $Z$ observable is 1. Mirroring also doubles the circuit depth, which amplifies the effect of noise.

In [1]:
import numpy as np
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp


def construct_ising_circuit(
    num_qubits: int,
    num_trotter_steps: int,
    rx_angle: float,
    barrier: bool = True,
) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)

    for _step in range(num_trotter_steps):
        circuit.rx(rx_angle, range(num_qubits))
        if barrier:
            circuit.barrier()
        for first_qubit in (1, 2):
            for idx in range(first_qubit, num_qubits, 2):
                # equivalent to Rzz(-pi/2):
                circuit.sdg([idx - 1, idx])
                circuit.cz(idx - 1, idx)
        if barrier:
            circuit.barrier()

    return circuit


def mirror_circuit(circuit: QuantumCircuit) -> QuantumCircuit:
    mirror_circ = circuit.copy_empty_like()
    mirror_circ.compose(circuit, inplace=True)
    mirror_circ.barrier()
    mirror_circ.compose(circuit.inverse(), inplace=True)
    mirror_circ.measure_active()
    return mirror_circ

In [2]:
num_qubits = 20
observables = [
    SparsePauliOp("I" * ((num_qubits - 1) - q) + "Z" + "I" * q)
    for q in range(num_qubits)
]

# Free parameters of the demonstration circuit: the number of Trotter steps
# sets the circuit depth, and rx_angle is the transverse-field rotation angle
# applied in each step (pi/4 makes the dynamics non-Clifford)
num_trotter_steps = 10
rx_angle = np.pi / 4

circuit = construct_ising_circuit(
    num_qubits, num_trotter_steps, rx_angle, barrier=False
)
mirrored_circuit = mirror_circuit(circuit)

In [3]:
mirrored_circuit.draw(
    "mpl", fold=-1, scale=0.3, idle_wires=False, measure_arrows=False
)

<Image src="/docs/images/tutorials/qdc26-profiling-tools/extracted-outputs/caa280da-0.avif" alt="Output of the previous code cell" />

## Step 2: Optimize for quantum hardware execution

### 2.1 Use alternative calibration if applicable

IBM&reg; QPUs are calibrated periodically for good performance across general use cases. Some workloads, however, benefit from instructions whose parameters are tuned differently. For more information, refer to [Specially calibrated instructions](/docs/guides/custom-calibration).

This tutorial uses an alternative calibration data set called `slow_gates_ac`. This calibration contains a slower `cz` gate and modulates the two-level system (TLS) bias to stabilize device errors over time, which benefits error mitigation methods that rely on a learned noise model. The `slow_gates_ac` calibration is available only on Heron QPUs. Use `slow_gates` instead if you are running on a Nighthawk QPU.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
# Use slow_gates_ac because this tutorial uses a noise-learning-based mitigation method
backend = service.backend("ibm_kingston", calibration_id="slow_gates_ac")

### 2.2 Run real-time device benchmarking

One of the most critical stages of transpilation is layout selection, which maps virtual qubits in a circuit to physical qubits on the target device. A good layout minimizes the number of SWAP operations needed and places the circuit on the best-performing qubits, so the quality of the initial layout can significantly affect the quality of your results.

Periodic benchmarking routines run on every IBM QPU to report the current performance of its qubits: relaxation times ($T_1$), coherence times ($T_2$), readout errors, and one- and two-qubit gate errors. The Qiskit transpiler uses these metrics during layout selection.

Superconducting qubits, however, can drift on shorter timescales, and transient defects (known as two-level systems) can temporarily degrade a qubit that performed well at calibration time. The transpiler cannot pick the best layout if it is working from stale information, so it can be worth spending some QPU time characterizing the device before selecting a layout.

Refer to the [Refresh backend properties with real-time benchmarking](/docs/tutorials/refresh-backend-properties-with-real-time-benchmarking) tutorial for a detailed walk-through.

The characterization suite builds several experiments, submits multiple jobs, and fits the results, which can take a few minutes in total. Enable INFO-level logging so that each stage prints its progress as it runs.

In [5]:
import sys
import logging

logger = logging.getLogger("qiskit_device_benchmarking")
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))

`BackendCharacterization` is a convenience class that runs the characterization experiments of your choice, analyzes the results, and returns an updated `Backend` object that you can pass to the transpiler.

You might see a warning that a backend was passed in as the mode while a batch is open. This is expected and can be safely ignored.

In [6]:
from qiskit_ibm_runtime import Batch
from qiskit_device_benchmarking.utilities.characterization_utils import (
    BackendCharacterization,
)

# The full suite is ["readout", "rb_1q", "rb_2q", "t1", "t2"].
# Run a subset of the experiments to save time.
experiments = ["readout", "t1", "t2"]
characterizer = BackendCharacterization(backend)
with Batch(backend=backend):
    jobs = characterizer.run_experiments(experiments=experiments)

base_primitive.get_mode_service_backend:WARNING:2026-11-03 17:36:59,925: A backend was passed in as the mode but a session context manager is open so this job will run inside this session/batch instead of in job mode.


Building readout experiments
Building T1 experiments
Building T2 experiments
Readout job submitted: dbl643m74qds73dpq210
T1 experiment submitted: ['dbl6449kbkrc73cn1u6g']
T2 (Hahn) experiment submitted: ['dbl6451kbkrc73cn1u80']


In [7]:
# Retrieve all job results and update the backend data
backend_updated = characterizer.update_backend()

### 2.3 Transpile to ISA

Transform the circuit and observables to match the backend's instruction set architecture (ISA) by using the refreshed device properties.

In [8]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(
    backend=backend_updated, optimization_level=3
)
isa_circuit = pm.run(mirrored_circuit)
isa_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

### 2.4 Visualize circuit timing

In addition to visualizing instructions on a circuit, you might want to visualize a circuit's scheduling to quickly spot idle time on qubits or inefficient instruction alignment. You can get the most detailed and accurate timing information from IBM Quantum Compute Service by using dry-run mode.

Refer to [Visualize circuit timing](/docs/guides/qiskit-runtime-circuit-timing) for more information.

Scheduled circuit timing is available only for Sampler and Executor jobs, so the following cell submits the circuit in dry-run mode by using Sampler. Jobs submitted in dry-run mode do not run on real QPUs and are not charged.

In [ ]:
from qiskit_ibm_runtime.executor_sampler import Sampler

sampler = Sampler(backend)
# Request timing information in the result metadata
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

# Specify dry_run=True to submit in dry-run mode
sampler_job = sampler.run([isa_circuit], dry_run=True)

In [10]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

In [ ]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# For large circuits, displaying the figure inline can hang your Jupyter session,
# so save it to an HTML file instead.
fig.write_html("scheduler_timing.html")

# For small circuits, uncomment the following line to display the figure inline.
# fig.show(renderer="notebook")

## Step 3: Apply error mitigation and test locally

### 3.1 Apply PEC with TREX

Selecting error mitigation methods that balance cost and accuracy for your specific experiment can be difficult. You can use the `select-error-mitigation` AI agent skill to analyze the available methods for your workload. The skill presents a summary report that explains whether each method was selected and why.

Refer to the [Qiskit AI skills](https://github.com/Qiskit/skills) GitHub repository for more information.

This tutorial uses PEC with TREX directly, so that you can follow along without an AI agent.

PEC is a technique for mitigating quantum noise in observable expectation value calculations. It produces an unbiased estimate of the expectation values by learning the noise of each layer of two-qubit gates, then executing an ensemble of circuits that, on average, apply the inverse of that noise. Pauli twirling is used to tailor the noise into a Pauli channel so that it can be learned and inverted efficiently.

TREX is a technique that mitigates measurement errors and is useful for estimating Pauli observable expectation values. It relies on measurement twirling, which diagonalizes the readout-error transfer matrix and makes it easy to invert. A rescaling term is learned from the diagonal noise channel by measuring twirled calibration circuits that are initialized in the zero state.

You can enable both PEC and TREX through Estimator options. This tutorial uses the client-side implementations of the Qiskit SamplerV2 and EstimatorV2 interfaces, introduced in `qiskit-ibm-runtime` v0.50.0. These client-side primitives perform most of the pre- and post-processing locally and delegate execution to the Executor primitive. Running on the client side provides greater transparency, faster debugging feedback, and finer control.

Because they implement the same API, the client-side primitives are largely drop-in replacements for the legacy server-side primitives; usually the only change needed is the import statement.

Enable INFO-level logging to see the progress of client-side pre- and post-processing.

In [12]:
logger = logging.getLogger("qiskit_ibm_runtime")
logger.setLevel(logging.INFO)

In [ ]:
from qiskit_ibm_runtime.executor_estimator import Estimator

estimator = Estimator(mode=backend)

In [14]:
# Enable TREX
estimator.options.resilience.measure_mitigation = True
# Enable PEC
estimator.options.resilience.pec_mitigation = True

### 3.2 Learn noise models

Both PEC and TREX require noise learning, which you can do with `NoiseLearnerV3`. The steps are as follows:

1. Find the unique two-qubit gate layers and measurement layers in the circuit, for the Pauli-Lindblad and TREX protocols, respectively.
2. Pass the layers to `NoiseLearnerV3` to learn the noise models.
3. Convert the learned two-qubit gate noise models to Pauli-Lindblad maps.

Refer to [Noise learning helper](/docs/guides/noise-learning) for more information about `NoiseLearnerV3`.

In [15]:
pubs = [(isa_circuit, isa_observables)]
# Specify types="all" to find layers for both Pauli-Lindblad and TREX learning
layers = estimator.find_unique_layers(pubs, types="all")
print(f"Number of unique layers found: {len(layers)}")

Number of unique layers found: 3


In [19]:
from qiskit_ibm_runtime import NoiseLearnerV3

learner = NoiseLearnerV3(backend)
learner.options.environment.job_tags = [
    "noise-learning",
    f"{len(layers)} layers",
]
noise_job = learner.run(layers)
print(f"NoiseLearner job id: {noise_job.job_id()}")

runtime.program_run:INFO:2026-11-03 17:39:54,921: Posting the API request.
runtime.program_run:INFO:2026-11-03 17:39:55,456: Payload size: 0 MB.


NoiseLearner job id: dbl65et8ft8s738sd1k0


In [20]:
noise_result = noise_job.result()
# to_pauli_lindblad_maps() ignores the measurement layers
pauli_lindblad_maps = noise_result.to_pauli_lindblad_maps()

Pass the learned noise model to Estimator for PEC.

In [22]:
estimator.options.resilience.layer_noise_model = zip(
    layers, pauli_lindblad_maps
)

### 3.3 Perform local simulation

With `qiskit-ibm-runtime` local testing mode, you can perform both noiseless and noisy simulations locally with minimal code changes. This is useful for verifying syntactic correctness, comparing the effects of different mitigation methods, and checking how well the learned noise model agrees with hardware results.

Local mode is designed to share largely the same interface as the QPU execution path. In most cases, you only need to change the backend passed when initializing the primitive and re-run the same code. Each of the following subsections re-initializes `Estimator`, so you can run them independently of each other.

Refer to [Local testing mode](/docs/guides/local-testing-mode) for more information.

#### 3.3.1 Cliffordize the circuit

The cost of general noisy simulation grows exponentially with the number of qubits, and most laptops cannot simulate more than about 32 qubits. Therefore, first convert the circuit to a Clifford circuit, which a stabilizer simulator can handle far more efficiently. The conversion rounds each rotation angle to a Clifford angle, so the simulated circuit approximates the original circuit rather than reproducing it.

In [24]:
from qiskit.transpiler import PassManager
from qiskit_ibm_runtime.transpiler.passes import ConvertISAToClifford

clifford_circuit = PassManager([ConvertISAToClifford()]).run(isa_circuit)
clifford_pubs = [(clifford_circuit, isa_observables)]

#### 3.3.2 Run a noiseless simulation

In [25]:
from qiskit_aer import AerSimulator

sim_estimator = Estimator(AerSimulator(method="stabilizer"))
sim_estimator.options.resilience_level = 0
job = sim_estimator.run(clifford_pubs)
ideal_result = job.result()
print("Ideal expectation values:")
print(ideal_result[0].data.evs)

estimator.run:INFO:2026-11-03 17:45:19,099: Starting pre-processing
prepare._build_quantum_program:INFO:2026-11-03 17:45:19,104: Processing pub 1/1 with MitigationTask.
estimator.run:INFO:2026-11-03 17:45:19,214: Submitting 1 pub to executor with 4096 total shots
post_processor_v0_1.estimator_v2_post_processor_v0_1:INFO:2026-11-03 17:45:21,059: Post-processing pub 1/1 (MitigationTask).


Ideal expectation values:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


#### 3.3.3 Run a noisy simulation with the learned Pauli-Lindblad model

<Admonition type="caution">
Do not use a fake backend or a Qiskit Aer noise model if you plan to use the learned Pauli-Lindblad noise model in your local simulation. Otherwise, the simulation applies the noise twice.
</Admonition>

In [26]:
sim_estimator = Estimator(AerSimulator(method="stabilizer"))
sim_estimator.options.resilience_level = 0
# Twirling is required for find_unique_layers to work
sim_estimator.options.twirling.enable_gates = True
sim_estimator.options.twirling.enable_measure = True
clifford_layers = sim_estimator.find_unique_layers(clifford_pubs, types="all")
sim_estimator.options.simulator.layer_noise_model = zip(
    clifford_layers, pauli_lindblad_maps
)

In [27]:
job = sim_estimator.run(clifford_pubs)
noisy_result = job.result()
print("Noisy expectation values with learned noise model:")
print(noisy_result[0].data.evs)

estimator.run:INFO:2026-11-03 17:45:32,319: Starting pre-processing
prepare._build_quantum_program:INFO:2026-11-03 17:45:32,324: Processing pub 1/1 with MitigationTask.
estimator.run:INFO:2026-11-03 17:45:32,495: Submitting 1 pub to executor with 4096 total shots
post_processor_v0_1.estimator_v2_post_processor_v0_1:INFO:2026-11-03 17:45:36,391: Post-processing pub 1/1 (MitigationTask).


Noisy expectation values with learned noise model:
[0.50585938 0.47119141 0.42626953 0.40380859 0.35693359 0.33203125
 0.3203125  0.28857422 0.28515625 0.27636719 0.26123047 0.28857422
 0.27001953 0.28125    0.30224609 0.33740234 0.38085938 0.42089844
 0.48144531 0.49169922]


#### 3.3.4 Run a noisy simulation with PEC and TREX

In [28]:
sim_estimator = Estimator(AerSimulator(method="stabilizer"))

# Twirling is required for find_unique_layers to work
sim_estimator.options.twirling.enable_gates = True
sim_estimator.options.twirling.enable_measure = True

# Enable PEC & TREX
sim_estimator.options.resilience.pec_mitigation = True
sim_estimator.options.resilience.measure_mitigation = True

# Reduce the number of shots to speed things up
sim_estimator.options.twirling.shots_per_randomization = 32
sim_estimator.options.twirling.num_randomizations = 16

clifford_layers = sim_estimator.find_unique_layers(clifford_pubs, types="all")

# This sets the noise model
sim_estimator.options.simulator.layer_noise_model = zip(
    clifford_layers, pauli_lindblad_maps
)
# This tells PEC which noise to cancel
sim_estimator.options.resilience.layer_noise_model = zip(
    clifford_layers, pauli_lindblad_maps
)

In [29]:
job = sim_estimator.run(clifford_pubs)
mitigated_result = job.result()
print("Expectation values with PEC and TREX applied:")
print(mitigated_result[0].data.evs)

estimator.run:INFO:2026-11-03 17:45:57,104: Starting pre-processing
prepare._build_quantum_program:INFO:2026-11-03 17:45:57,112: Processing pub 1/1 with PEC.
trex.apply_trex:INFO:2026-11-03 17:45:57,339: TREX calibration circuit added (16 randomizations).
estimator.run:INFO:2026-11-03 17:45:57,340: Submitting 2 pubs to executor with 51712 total shots
post_processor_v0_1.estimator_v2_post_processor_v0_1:INFO:2026-11-03 17:46:48,840: Post-processing pub 1/1 (PEC).


Expectation values with PEC and TREX applied:
[0.65647631 0.69508478 0.67541152 0.75698875 0.73556673 0.77556676
 0.59293664 0.79267548 0.52579479 0.74500773 0.61135387 0.75841915
 0.60962437 0.66695697 0.72544458 0.76858551 0.61339628 0.68928036
 0.75124923 0.79362096]


### 3.4 Estimate usage

The main use case of dry-run mode is usage estimation. Unlike local execution mode, dry-run mode submits the job to IBM Quantum Compute Service, which returns an accurate usage estimate but **random result data**. Dry-run jobs do not run on real QPUs and do not consume any QPU time.

Dry-run jobs are not available on the Open or Pay-As-You-Go plans.

Refer to [Dry-run jobs](/docs/guides/dry-run-jobs) for more information.

In [ ]:
dry_run_job = estimator.run([(isa_circuit, isa_observables)], dry_run=True)
# Wait for the job to finish to get an accurate estimate
dry_run_job.result()
print(f">>> Usage is {dry_run_job.usage()} seconds")

estimator.run:INFO:2026-11-03 17:50:01,081: Starting pre-processing
prepare._build_quantum_program:INFO:2026-11-03 17:50:01,097: Processing pub 1/1 with PEC.
trex.apply_trex:INFO:2026-11-03 17:50:01,344: TREX calibration circuit added (64 randomizations).
estimator.run:INFO:2026-11-03 17:50:01,345: Submitting 2 pubs to executor with 413696 total shots
runtime.program_run:INFO:2026-11-03 17:50:02,143: Posting the API request.
runtime.program_run:INFO:2026-11-03 17:50:03,189: Payload size: 0 MB.
post_processor_v0_1.estimator_v2_post_processor_v0_1:INFO:2026-11-03 17:50:53,777: Post-processing pub 1/1 (PEC).


Usage is 126 seconds


## Step 4: Run on hardware

### 4.1 Initialize a session and run calibration

Calibrating a QPU requires optimizing many parameters that define the electrical signals driving the quantum gates and operations. Calibration jobs run throughout the day in between user jobs, but not during an active session. Because these parameters can drift within a short period of time, it might be worth triggering calibration at the start of a session, or periodically during a long-running session, if you want the best possible results.

Currently, only readout calibration is supported, and it takes approximately five minutes of wall-clock time.

In [33]:
from qiskit_ibm_runtime import Session, Calibrator

# Set to False to skip the session and calibration and run in job mode instead
run_calibration = True

mode = backend
if run_calibration:
    mode = Session(backend)
    calibrator = Calibrator(mode)
    calibrator.run().result()

runtime.program_run:INFO:2026-11-03 17:52:49,826: Posting the API request.
runtime.program_run:INFO:2026-11-03 17:52:50,428: Payload size: 0 MB.


### 4.2 Run on hardware with Estimator

In [34]:
estimator = Estimator(mode)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.layer_noise_model = zip(
    layers, pauli_lindblad_maps
)

job = estimator.run([(isa_circuit, isa_observables)])
print(f"Estimator job id: {job.job_id()}")

if run_calibration:
    # Don't forget to close the session if you're using one!
    mode.close()

estimator.run:INFO:2026-11-03 17:58:14,304: Starting pre-processing
prepare._build_quantum_program:INFO:2026-11-03 17:58:14,311: Processing pub 1/1 with PEC.
trex.apply_trex:INFO:2026-11-03 17:58:14,535: TREX calibration circuit added (64 randomizations).
estimator.run:INFO:2026-11-03 17:58:14,536: Submitting 2 pubs to executor with 413696 total shots
runtime.program_run:INFO:2026-11-03 17:58:14,973: Posting the API request.
runtime.program_run:INFO:2026-11-03 17:58:17,405: Payload size: 0 MB.


Estimator job id: dbl6e29kbkrc73cn2mr0


In [35]:
result = job.result()
print("Expectation values from QPU run:")
print(result[0].data.evs)

post_processor_v0_1.estimator_v2_post_processor_v0_1:INFO:2026-11-03 18:04:25,471: Post-processing pub 1/1 (PEC).


Expectation values from QPU run:
[0.87562758 0.77703872 0.71939924 0.69740788 0.67939584 0.70667221
 0.72209841 0.73412873 0.66495252 0.67249968 0.65672049 0.6431414
 0.66205351 0.6598997  0.68112856 0.69308645 0.71226384 0.73133117
 0.78972661 0.89014635]


### 4.3 Inspect execution spans

When you submit circuits with multiple configurations, such as with gate twirling, IBM Quantum Compute Service does not necessarily execute all circuit configurations at once. Instead, it groups them into chunks and executes one chunk per shot loop. A shot loop is a single round of execution on the quantum hardware, in which every circuit in the chunk is run for the requested number of shots.

All jobs submitted to a real QPU go through some classical pre-processing, such as translation into pulses that can be sent to the hardware. Because this pre-processing can be done in parallel, the service dequeues and starts multiple jobs at a time. The QPU, however, can only process one unit of work at a time. This means that if your job generates multiple chunks, its chunks might be interleaved with those of other users' jobs, and each chunk might have to wait for its turn on the QPU.

QPU execution timing data for each chunk is reported in the job result metadata. The following visualization shows whether there are significant gaps between the executions of the chunks in your job. If the gaps are wider than your experiment can tolerate, you might want to re-run the job, or use [session execution mode](/docs/guides/execution-modes#session-mode), which gives your workload exclusive access to the QPU.

Refer to [View execution spans](/docs/guides/sampler-input-output#view-execution-spans) for more information.

In [36]:
chunk_timing = result.metadata["executor"].chunk_timing

print(f"{len(chunk_timing)} chunks were executed.")
chunk_1 = chunk_timing[0]
print(f"The first chunk contains {len(chunk_1.parts)} parts.")
for idx, part in enumerate(chunk_1.parts):
    print(
        f"    Part {idx} contains {part.size} configurations from circuit {part.idx_item}"
    )

8 chunks were executed.
The first chunk contains 2 parts.
    Part 0 contains 100 configurations from circuit 0
    Part 1 contains 1 configurations from circuit 1


In [37]:
from qiskit_ibm_runtime.visualization import draw_chunk_timings

draw_chunk_timings(chunk_timing)

## Next steps

<Admonition type="tip" title="Recommendations">
- Compare the hardware expectation values from Step 4 with the noiseless, unmitigated, and mitigated simulation results from Step 3.3. Because the circuit is a mirror circuit, every ideal value is 1, so you can directly see how closely the learned noise model predicts the hardware behavior and how much PEC and TREX improve the results.
- Re-run the experiment with and without the refreshed device properties (Step 2.2), the `slow_gates_ac` calibration (Step 2.1), or session calibration (Step 4.1) to measure how much each tool contributes for your workload.
- Try a cheaper error mitigation method such as zero-noise extrapolation (ZNE), and use local simulation and dry-run usage estimates to compare its cost and accuracy against PEC. The [`select-error-mitigation`](https://github.com/Qiskit/skills) AI agent skill can help guide this choice.
- Explore the full characterization suite, including randomized benchmarking, in the [Refresh backend properties with real-time benchmarking](/docs/tutorials/refresh-backend-properties-with-real-time-benchmarking) tutorial.
- Learn more about the client-side primitives and the Executor in [Migrate to client-side primitives](/docs/guides/migrate-to-client-side-primitives).
- Apply these tools to your own experiments, starting with the ones that address your biggest source of uncertainty, whether that is device quality, noise, cost, or execution timing.
</Admonition>